In [ ]:
from fast_gnn_benchmark.data.dataset.ogbl import FixLinkPropPredDataset

ds = FixLinkPropPredDataset(name="ogbl-ppa", root="datasets/ogbl/")


In [2]:
from fast_gnn_benchmark.schemas.dataset_models import SplitType
from torch_geometric.transforms import ToSparseTensor
from torch_geometric.utils import degree, subgraph
from tqdm import tqdm

from fast_gnn_benchmark.data.utils import to_undirected
import os
from collections.abc import Callable, Iterator
from math import ceil
from typing import Any
from torch_geometric.data import Data, InMemoryDataset

import numpy as np
import pandas as pd
import torch

class LinkLoader:
    def __init__(
        self,
        dataset: Any,
        batch_size: int,
        mask_loss_edges: bool = True,
        split_type: SplitType = SplitType.TRAIN,
        max_iterations: int = 3,
        negative_sampling_ratio: float = 0.5,
        on_device=True
    ):

        if on_device:
            self.device = torch.accelerator.current_accelerator() or torch.device("cpu")
        else:
            self.device = torch.device("cpu")
        
        self.data = dataset.data.to(self.device)
        self.num_nodes = dataset.num_nodes
        self.batch_size = batch_size
        self.mask_loss_edges = mask_loss_edges
        self.split_type = split_type
        self.max_iterations = max_iterations
        self.negative_per_batch = int(batch_size * negative_sampling_ratio)
        self.positive_per_batch = batch_size - self.negative_per_batch

        self.to_sparse_tensor = ToSparseTensor()

        match split_type:
            case SplitType.TRAIN:
                self.positive_edges, self.non_negative_edges_ids = self.cannonize_positive_edges(
                    dataset, remove_self_loops=True
                )

            case SplitType.VAL:
                splits = dataset.get_edge_split()
                positive_edges = splits["valid"]["edge"].T
                negative_edges = splits["valid"]["edge_neg"].T
                self.target_edges = torch.cat([positive_edges, negative_edges], dim=1).to(self.device)
                self.labels = torch.cat(
                    [torch.ones(positive_edges.shape[1]), torch.zeros(negative_edges.shape[1])], dim=0
                ).to(self.device)
            case SplitType.TEST:
                splits = dataset.get_edge_split()
                positive_edges = splits["test"]["edge"].T
                negative_edges = splits["test"]["edge_neg"].T
                self.target_edges = torch.cat([positive_edges, negative_edges], dim=1).to(self.device)
                self.labels = torch.cat(
                    [torch.ones(positive_edges.shape[1]), torch.zeros(negative_edges.shape[1])], dim=0
                ).to(self.device)
            case _:
                raise ValueError(f"Invalid split type: {split_type}")

    def cannonize_positive_edges(self, dataset: Any, remove_self_loops: bool = True) -> tuple[torch.Tensor, torch.Tensor]:
        positive_edges = dataset.get_edge_split()["train"]["edge"].T  # [2, n]
        positive_edges = torch.sort(positive_edges, dim=0).values
        if remove_self_loops:
            non_negative_edges = torch.cat([positive_edges, torch.arange(self.num_nodes).repeat(2, 1)], dim=1)
        else:
            non_negative_edges = positive_edges

        non_negative_edges_ids = non_negative_edges[0, :] * self.num_nodes + non_negative_edges[1, :]

        positive_edges = positive_edges.to(self.device)
        non_negative_edges_ids = non_negative_edges_ids.unique().to(self.device)

        return positive_edges, non_negative_edges_ids

    def rejection_sampling_negative_edges(self) -> torch.Tensor:
        candidates = torch.empty((2, 0), dtype=torch.int64, device=self.device)

        for _ in range(self.max_iterations):
            negative_candidates = torch.randint(0, self.num_nodes, (2, self.negative_per_batch), device=self.device)
            src = torch.minimum(negative_candidates[0, :], negative_candidates[1, :])
            dst = torch.maximum(negative_candidates[0, :], negative_candidates[1, :])

            candidate_edges_ids = src * self.num_nodes + dst

            idx = torch.searchsorted(self.non_negative_edges_ids, candidate_edges_ids)
            is_positive = torch.logical_and(
                idx < len(self.non_negative_edges_ids),
                self.non_negative_edges_ids[idx] == candidate_edges_ids,
            )

            candidates = torch.cat([candidates, negative_candidates[:, ~is_positive]], dim=1)

            if candidates.shape[1] >= self.negative_per_batch:
                return candidates[:, : self.negative_per_batch]
        return candidates[:, : self.negative_per_batch]

    def __len__(self) -> int:
        if self.split_type == SplitType.TRAIN:
            return self.positive_edges.shape[1] // self.positive_per_batch

        return self.target_edges.shape[1] // self.batch_size

    def __iter__(self) -> Iterator[Data]:
        return self.get_iterator()

    def get_iterator(self) -> Iterator[Data]:
        if self.split_type == SplitType.TRAIN:
            for start_idx in range(0, self.positive_edges.shape[1], self.positive_per_batch):
                end_idx = start_idx + self.positive_per_batch

                positive_edges = self.positive_edges[:, start_idx:end_idx]
                negative_edges = self.rejection_sampling_negative_edges()
                target_edges = torch.cat([positive_edges, negative_edges], dim=1)
                labels = torch.cat([torch.ones(positive_edges.shape[1]), torch.zeros(negative_edges.shape[1])], dim=0)
                if self.mask_loss_edges:
                    edge_index = torch.cat(
                        [self.positive_edges[:, :start_idx], self.positive_edges[:, end_idx:]], dim=1
                    )
                    data = Data(
                        x=self.data.x,
                        edge_index=to_undirected(edge_index),
                        target_edges=target_edges,
                        y=labels,
                    )
                else:
                    data = self.data.clone()
                    data.target_edges = target_edges
                    data.y = labels

                data = self.to_sparse_tensor(data)
                data.edge_index = data.adj_t

                yield data

        else:
            for start_idx in range(0, self.target_edges.shape[1], self.batch_size):
                end_idx = start_idx + self.batch_size
                target_edges = self.target_edges[:, start_idx:end_idx]
                labels = self.labels[start_idx:end_idx]
                data = self.data.clone()
                data.target_edges = target_edges
                data.y = labels

                data = self.to_sparse_tensor(data)
                data.edge_index = data.adj_t

                yield data



train_loader = LinkLoader(
    dataset=ds,
    batch_size = 131072, # 1000000
    mask_loss_edges = True,
    split_type = SplitType.TRAIN,
    max_iterations = 3,
    negative_sampling_ratio = 0.5,
)

val_loader = LinkLoader(
    dataset=ds,
    batch_size = 131072,
    split_type = SplitType.VAL,
)

test_loader = LinkLoader(
    dataset=ds,
    batch_size = 131072,
    split_type = SplitType.TEST,
)


print(train_loader)

                          


/tmp/ipykernel_3488615/1413689383.py:34: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  self.data = dataset.data.to(self.device)
/home/infres/clwang/fast-gnn-benchmark/.venv/lib/python3.11/site-packages/torch/serialization.py:1812: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  result = unpickler.load()


In [3]:
from tqdm import tqdm

for batch in tqdm(train_loader):
    pass
    
    

324it [00:22, 14.64it/s]                                                                                                                                  


In [4]:
print(len(train_loader))
print(len(val_loader))
print(len(test_loader))

323
69
46


In [5]:
from fast_gnn_benchmark.schemas.model import GNNParameters, ArchitectureType
from fast_gnn_benchmark.models.backbones import load_backbone


architecture_params = dict(
    architecture_type = ArchitectureType.GCN,
    input_dim = 58, 
    num_layers = 3,
    hidden_dim = 128,
    output_dim = 128,
    dropout = 0.1,
    use_input_projection = False,
    use_output_projection = False, 
    use_residual = False,
    use_layer_norm = False,
    use_batch_norm = False,
    
)

architecture_params = GNNParameters(**architecture_params)


gnn = load_backbone(architecture_params)



In [6]:

import torch
import torch.nn.functional as F

class Hadamard_MLPPredictor(torch.nn.Module):
    def __init__(self, h_feats, dropout, layer=2, res=False, norm=False, scale=False, act='relu'):
        super().__init__()
        self.lins = torch.nn.ModuleList()
        self.lins.append(torch.nn.Linear(h_feats, h_feats))
        for _ in range(layer - 2):
            self.lins.append(torch.nn.Linear(h_feats, h_feats))
        self.lins.append(torch.nn.Linear(h_feats, 1))
        self.dropout = dropout
        self.res = res
        self.scale = scale
        if scale:
            self.scale_norm = nn.LayerNorm(h_feats)
        self.norm = norm
        if norm:
            self.norms = torch.nn.ModuleList()
            for _ in range(layer - 1):
                self.norms.append(nn.LayerNorm(h_feats))
        if act == 'relu':
            self.act = F.relu
        elif act == 'gelu':
            self.act = F.gelu
        elif act == 'silu':
            self.act = F.silu
        else:
            raise ValueError('Activation function not supported')

    def forward(self, x_i, x_j):
        x = x_i * x_j
        if self.scale:
            x = self.scale_norm(x)
        ori = x
        for i in range(len(self.lins) - 1):
            x = self.lins[i](x)
            if self.res:
                x += ori
            if self.norm:
                x = self.norms[i](x)
            x = self.act(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lins[-1](x)
        return x.squeeze()


predictor = Hadamard_MLPPredictor(
    h_feats = 128, 
    dropout = 0.2,
    layer = 2,
    res=True,
    norm=False,
    scale=False,
    act="relu"
)


class ModelWrapper(torch.nn.Module):
    def __init__(self, gnn, predictor):
        super().__init__()
        self.gnn = gnn
        self.predictor = predictor

    def forward(self, x):
        pass


model_wrapper = ModelWrapper(gnn, predictor)



optimizer = torch.optim.Adam(model_wrapper.parameters(), lr=0.001)

model_wrapper


ModelWrapper(
  (gnn): GNNStack(
    (conv_layers): ModuleList(
      (0): GCNConv(58, 128)
      (1-2): 2 x GCNConv(128, 128)
    )
  )
  (predictor): Hadamard_MLPPredictor(
    (lins): ModuleList(
      (0): Linear(in_features=128, out_features=128, bias=True)
      (1): Linear(in_features=128, out_features=1, bias=True)
    )
  )
)

In [7]:
from tqdm import tqdm

def train_loop(loader, optimizer, model_wrapper, device):
    total_loss = 0
    
    for batch in tqdm(loader):
        
        batch = batch.to(device)
        state = model_wrapper.gnn(batch.x.float(), batch.edge_index)
        src_edge = batch.target_edges[0]
        dst_edge = batch.target_edges[1]
        
        src_state = state[src_edge]
        dst_state = state[dst_edge]

        pred = model_wrapper.predictor(src_state, dst_state)

        loss = F.binary_cross_entropy_with_logits(pred, batch.y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.cpu().item()

    return total_loss / len(loader)

def val_loop(loader, model_wrapper, device):
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(loader):

            batch = batch.to(device)


            state = model_wrapper.gnn(batch.x.float(), batch.edge_index)
            src_edge = batch.target_edges[0]
            dst_edge = batch.target_edges[1]
    
            src_state = state[src_edge]
            dst_state = state[dst_edge]
    
            pred = model_wrapper.predictor(src_state, dst_state)
    
            loss = F.binary_cross_entropy_with_logits(pred, batch.y)
            total_loss += loss.cpu().item()

    return total_loss / len(loader)



device = "cuda"
model_wrapper = model_wrapper.to(device)

for i in range(10):
    print("epoch", i)
    train_loss = train_loop(train_loader, optimizer, model_wrapper, device)
    print("train loss", train_loss)
    val_loss = val_loop(val_loader, model_wrapper, device)
    print("val loss", val_loss)


test_loss = val_loop(test_loader, model_wrapper, device)

print("test loss", test_loss)



        
        

epoch 0


324it [01:59,  2.72it/s]                                                                                                                                  


train loss 0.37425639275057765


70it [00:13,  5.29it/s]                                                                                                                                   


val loss 0.2464244404564733
epoch 1


...

train loss 0.12978543311172963


70it [00:13,  5.29it/s]                                                                                                                                   


val loss 0.12498226805009703


47it [00:08,  5.29it/s]                                                                                                                                   

test loss 0.13010773953536284
